<a href="https://colab.research.google.com/github/saifxyzyz/DermaGemma/blob/main/notebooks/gemma4good_RAG_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install sentence-transformers rank_bm25 faiss-cpu groq openai timm transformers openai transformers accelerate timm tokenizers

In [5]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from typing import Dict, List

In [6]:
KNOWLEDGE_BASE = [
    # 1. Post-Inflammatory Hyperpigmentation (PIH)
    {'id':'pih_def','pathology':'Post-Inflammatory Hyperpigmentation','type':'definition','source':'JCAD','text':'PIH is an acquired hypermelanosis occurring after cutaneous inflammation or injury. It is significantly more prevalent and severe in skin-of-color patients (Fitzpatrick IV-VI).'},
    {'id':'pih_feat','pathology':'Post-Inflammatory Hyperpigmentation','type':'feature','source':'JCAD','text':'Epidermal PIH appears as tan, brown, or dark brown macules. Dermal PIH has a blue-gray appearance. It worsens with UV exposure and persistent inflammation.'},
    {'id':'pih_phr','pathology':'Post-Inflammatory Hyperpigmentation','type':'phrasing','source':'Clinical Guidelines','text':'Example phrasing: "Hyper-pigmented macules noted in areas of previous acne lesions, consistent with post-inflammatory hyperpigmentation."'},
    {'id':'pih_pit','pathology':'Post-Inflammatory Hyperpigmentation','type':'pitfall','source':'JCAD','text':'Pitfall: Deeper dermal pigmentation does not respond to topical tyrosinase inhibitors (like hydroquinone) and may be permanent if untreated.'},

    # 2. Acne Vulgaris
    {'id':'acne_def','pathology':'Acne Vulgaris','type':'definition','source':'Taylor & Kelly','text':'Acne vulgaris in SOC is characterized by a high frequency of post-inflammatory hyperpigmentation. Pathogenesis is similar across races, but sequelae differ.'},
    {'id':'acne_feat','pathology':'Acne Vulgaris','type':'feature','source':'Taylor & Kelly','text':'Chief complaint is often "dark marks" rather than active pustules. Pomade acne (from hair products) is a frequent precipitating factor in SOC.'},
    {'id':'acne_pit','pathology':'Acne Vulgaris','type':'pitfall','source':'Taylor & Kelly','text':'Pitfall: Avoid aggressive drying topicals; irritation in dark skin can lead to worsening PIH and "keloidal" scarring.'},

    # 3. Atopic Dermatitis (Eczema)
    {'id':'ad_def','pathology':'Atopic Dermatitis','type':'definition','source':'SUNY Downstate','text':'Eczema in darker skin often lacks the "classic" bright redness. It is frequently underdiagnosed due to atypical presentation.'},
    {'id':'ad_feat','pathology':'Atopic Dermatitis','type':'feature','source':'SUNY Downstate','text':'Erythema may appear purple, gray, or dark brown. Chronic cases show significant "ashiness," lichenification (thickening), and follicular prominence.'},
    {'id':'ad_pit','pathology':'Atopic Dermatitis','type':'pitfall','source':'EverydayHealth','text':'Pitfall: Severity scoring tools often underestimate inflammation in dark skin because redness is less visible.'},

    # 4. Seborrheic Dermatitis
    {'id':'sd_def','pathology':'Seborrheic Dermatitis','type':'definition','source':'DermNet','text':'A common inflammatory condition causing scaling. In SOC, it often presents as "petaloid" (flower-shaped) hypopigmented patches.'},
    {'id':'sd_feat','pathology':'Seborrheic Dermatitis','type':'feature','source':'DermNet','text':'Scalp involvement (dandruff) is common. In the face, look for scaling in the eyebrows and nasolabial folds.'},

    # 5. Tinea Corporis (Ringworm)
    {'id':'tinea_def','pathology':'Tinea Corporis','type':'definition','source':'StatPearls','text':'Superficial fungal infection. In dark skin, the "ring" may have a hyperpigmented, scaly border with central clearing or hypopigmentation.'},

    # 6. Keloids
    {'id':'kel_def','pathology':'Keloids','type':'definition','source':'Taylor & Kelly','text':'Keloids are overgrowths of dense fibrous tissue that invade healthy tissue beyond the original injury site. High familial susceptibility in SOC.'},
    {'id':'kel_feat','pathology':'Keloids','type':'feature','source':'Taylor & Kelly','text':'Firm, rubbery, shiny lesions. Common on earlobes, chest, and back. Often painful or pruritic (itchy).'},
    {'id':'kel_pit','pathology':'Keloids','type':'pitfall','source':'Taylor & Kelly','text':'Pitfall: Frequently confused with hypertrophic scars; keloids do not regress spontaneously and extend beyond the wound boundary.'},

    # 7. Dermatosis Papulosa Nigra (DPN)
    {'id':'dpn_def','pathology':'Dermatosis Papulosa Nigra','type':'definition','source':'AccessMedicine','text':'DPN are benign, small, darkly pigmented papules (flesh moles) appearing primarily on the face and neck of SOC individuals.'},
    {'id':'dpn_feat','pathology':'Dermatosis Papulosa Nigra','type':'feature','source':'AccessMedicine','text':'Chronic and progressive with age. Genetic predilection in 50% of cases. Histology is similar to seborrheic keratosis.'},

    # 8. Pseudofolliculitis Barbae
    {'id':'pfb_def','pathology':'Pseudofolliculitis Barbae','type':'definition','source':'DermNet','text':'"Razor bumps" caused by curly hair re-entering the skin. Predominantly affects men with coarse, curly hair.'},

    # 9. Vitiligo
    {'id':'vit_def','pathology':'Vitiligo','type':'definition','source':'StatPearls','text':'Autoimmune destruction of melanocytes. Depigmented patches are starkly visible and highly stigmatizing in dark-skinned populations.'},

    # 10. Pityriasis Alba
    {'id':'pa_def','pathology':'Pityriasis Alba','type':'definition','source':'Dermatology Advisor','text':'A low-grade form of eczema. Presents as round, scaly, light (hypopigmented) patches on children\'s faces.'},
    {'id':'pa_feat','pathology':'Pityriasis Alba','type':'feature','source':'Dermatology Advisor','text':'Patches become more apparent in summer as surrounding skin tans, increasing contrast. Usually resolves by puberty.'},

    # 11. Melasma
    {'id':'mel_def','pathology':'Melasma','type':'definition','source':'StatPearls','text':'Acquired hyperpigmentation, often hormonal. Presents as symmetrical brown/gray patches on the face.'},
    {'id':'mel_feat','pathology':'Melasma','type':'feature','source':'StatPearls','text':'UV and visible light are major drivers. Tinted mineral sunscreens (iron oxides) are recommended for SOC patients.'},

    # 12. Lichen Planus
    {'id':'lp_def','pathology':'Lichen Planus','type':'definition','source':'Mayo Clinic','text':'Inflammatory condition affecting skin and mucous membranes. Characterized by the "6 Ps": Planar, Purple, Polygonal, Pruritic, Papules, and Plaques.'},
    {'id':'lp_pit','pathology':'Lichen Planus','type':'pitfall','source':'Mayo Clinic','text':'Pitfall: In dark skin, purple hues may appear dark brown; look for "Wickham striae" (fine white lines) on the surface.'},

    # 13. Traction Alopecia
    {'id':'ta_def','pathology':'Traction Alopecia','type':'definition','source':'DermNet','text':'Hair loss due to chronic tension (tight braids/extensions). Early stage is reversible; late stage leads to permanent scarring.'},

    # 14. Acne Keloidalis Nuchae (AKN)
    {'id':'akn_def','pathology':'Acne Keloidalis Nuchae','type':'definition','source':'Taylor & Kelly','text':'Folliculitis that evolves into keloid-like papules on the posterior neck/occipital scalp. Almost exclusive to darker-pigmented men with curly hair.'},
    {'id':'akn_feat','pathology':'Acne Keloidalis Nuchae','type':'feature','source':'Taylor & Kelly','text':'Can lead to scarring alopecia (permanent hair loss). Early treatment with Class I steroids is the medical standard.'},

    # 15. Acanthosis Nigricans
    {'id':'an_def','pathology':'Acanthosis Nigricans','type':'definition','source':'StatPearls','text':'Velvety, dark thickening in skin folds (neck, armpits). Primarily a cutaneous marker of insulin resistance or obesity.'},
    {'id':'an_feat','pathology':'Acanthosis Nigricans','type':'feature','source':'StatPearls','text':'Velvety texture is more diagnostic than color alone. Poorly defined borders. Screening for diabetes (A1c) is recommended.'},
]

print('KB size', len(KNOWLEDGE_BASE))

KB size 31


In [7]:
# --- 3. RETRIEVAL ENGINE SETUP ---
# Running completely inside Colab cloud memory to preserve your local disk space
embedder = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')

def _tok(text):
    return text.lower().split()

corpus_texts = [s['text'] for s in KNOWLEDGE_BASE]
tokenized_corpus = [_tok(txt) for txt in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

embeddings = embedder.encode(corpus_texts, normalize_embeddings=True).astype('float32')
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
class DermatologyRetriever:
    def __init__(self, label_threshold=0.3, top_n_total=3, rrf_k=60):
        self.label_threshold = label_threshold
        self.top_n_total = top_n_total
        self.rrf_k = rrf_k

    def retrieve(self, predicted_labels: Dict[str, float]) -> List[dict]:
        active = [(l, p) for l, p in predicted_labels.items() if p >= self.label_threshold]
        active = sorted(active, key=lambda x: -x[1])

        if not active:
            return [s for s in KNOWLEDGE_BASE if s['pathology'] == 'Normal']

        active_labels = {l.lower().replace('_', ' ') for l, _ in active}
        label_to_confidence = {l.lower().replace('_', ' '): p for l, p in active}

        candidate_idxs = [
            i for i, s in enumerate(KNOWLEDGE_BASE)
            if s['pathology'].lower().replace('_', ' ') in active_labels
        ]

        if not candidate_idxs:
            return []

        query = ' '.join(l for l, _ in active) + ' skin of color presentation features guidelines'

        # Dense Embedding Vector Search
        e = embedder.encode([query], normalize_embeddings=True).astype('float32')
        _, dense_ix = faiss_index.search(e, len(KNOWLEDGE_BASE))
        dense_order = dense_ix[0].tolist()

        # Lexical BM25 Search
        sc = bm25.get_scores(_tok(query))
        bm25_order = np.argsort(sc)[::-1].tolist()

        scored = []
        for i in candidate_idxs:
            rrf_score = 0.0
            if i in dense_order:
                rrf_score += 1.0 / (self.rrf_k + dense_order.index(i) + 1)
            if i in bm25_order:
                rrf_score += 1.0 / (self.rrf_k + bm25_order.index(i) + 1)

            pathology_name = KNOWLEDGE_BASE[i]['pathology'].lower().replace('_', ' ')
            rrf_score *= (1.0 + label_to_confidence.get(pathology_name, 1.0))

            type_boost = {'phrasing': 0.15, 'feature': 0.12, 'definition': 0.05}
            rrf_score += type_boost.get(KNOWLEDGE_BASE[i]['type'], 0.0)

            scored.append((i, rrf_score))

        scored.sort(key=lambda x: -x[1])
        return [KNOWLEDGE_BASE[i] for i, _ in scored[:self.top_n_total]]

retriever = DermatologyRetriever()
print("Dermatology RAG System Initialized for OpenRouter Pipelines.")

Dermatology RAG System Initialized for OpenRouter Pipelines.


# Load Gemma 4 E4B

In [1]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate timm tokenizers

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-85n5_yzs
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-85n5_yzs
  Resolved https://github.com/huggingface/transformers.git to commit 5d9bce2548fc6fa70d0e38e7999a29bedaa4feeb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForImageTextToText, AutoTokenizer
from google.colab import userdata

# 1. Authenticate with Hugging Face Hub using your token secret
hf_token = userdata.get('HF_TOKEN')

model_id = "google/gemma-4-e4b-it"

# 2. Load Tokenizer and the correct Multimodal Model architecture class
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16, # Keeps memory footprint lightweight
    token=hf_token
)

# Extract the base text word embedding layer from the text configuration config
gemma_embeddings = model.get_input_embeddings()
text_dim = model.config.text_config.hidden_size # Access hidden size through nested text config
print(f"✅ Gemma 4 E4B Loaded. Hidden Dimension: {text_dim}")

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [12]:
# --- SECURE OPENROUTER API KEY SETUP ---
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

print('OpenRouter API Key Active:', bool(OPENROUTER_API_KEY))

OpenRouter API Key Active: True


In [13]:
import os
import json
from openai import OpenAI

In [23]:
import os
import json
from openai import OpenAI

# --- SECURE OPENROUTER API KEY SETUP ---
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

print('OpenRouter API Key Active:', bool(OPENROUTER_API_KEY))

def generate_dermatology_report(vit_outputs: dict) -> str:
    if not OPENROUTER_API_KEY:
        return "Error: Please set your OPENROUTER_API_KEY in Colab Secrets."

    # Point the client to OpenRouter's architecture base URL
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    # 1. Run the RAG Retrieval engine we configured earlier
    matched_guidelines = retriever.retrieve(vit_outputs)

    # 2. Format Context Strings for the prompt
    context_str = ""
    for idx, doc in enumerate(matched_guidelines):
        context_str += f"\n[{idx+1}] Source: {doc['source']} ({doc['type']}) - Context: {doc['text']}\n"

    # 3. Create System Prompt instructions
    system_instruction = """You are an experienced clinical mellanin rich dermatologist drafting an evaluation report for Skin of Color (Fitzpatrick Types IV-VI).

STRICT GENERATION RULES:
1. Mention ONLY pathologies explicitly provided in the VISION MODEL OUTPUTS list. Do not introduce diagnostic labels or conditions from the retrieved context if they are not in the predicted list.
2. The retrieved clinical knowledge base context is for STYLE, MORPHOLOGICAL DESCRIPTION, and EVIDENCE-BASED MANAGEMENT reference only — use it to write about predicted conditions, not to add unpredicted diagnoses.
3. For any predicted abnormality with an output probability < 0.5, use clinical hedging language ("suggestive of", "possible early-stage", "cannot be excluded from differential diagnosis").
4. If a severe condition is absent or below the threshold, use explicit clinical negation rules where appropriate ("no clinical evidence of keloidal tissue", "no deep ulceration identified").

Generate exactly two distinct sections:
1. CLINICAL EXAMINATION & FINDINGS — A systematic morphological description of the skin layers (epidermal pigmentation changes, dermal texture alterations, follicular prominence, scaling/borders, and surrounding skin matrix characteristics).
2. IMPRESSION & MANAGEMENT PLAN — A synthesized diagnostic conclusion mapping directly to the vision classification confidence, followed by evidence-based direct next steps or specialist pitfalls strictly based on the provided reference guidelines."""

    user_prompt = f"""
    Vision Model Outputs (ViT Classifier): {json.dumps(vit_outputs)}

    Retrieved Clinical Knowledge Base Context:
    {context_str}

    Generate a formatted diagnostic report containing:
    1. Primary Suspected Condition (include classification confidence)
    2. Visual Clinical Presentation Analysis (referencing why it matches skin of color markers)
    3. Recommended Direct Actions / Clinical Guidelines
    4. Diagnostic Pitfalls or monitoring warnings.
    """

    # 4. Generate with Free Gemma 4 Engine on OpenRouter
    try:
        completion = client.chat.completions.create(
            # Calling the specific free deployment string for Gemma 4
            model="google/gemma-4-26b-a4b-it:free",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.15
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Report generation failed: {str(e)}"

OpenRouter API Key Active: True


In [22]:
# Test predictions coming out from your vision layer
mock_vit_predictions = {
    "Keloids": 0.85,
    "Post-Inflammatory Hyperpigmentation": 0.38
}

final_report = generate_dermatology_report(mock_vit_predictions)
print(final_report)

**CLINICAL EXAMINATION & FINDINGS**

The examination reveals localized, elevated cutaneous lesions characterized by a firm, rubbery texture and a notable shiny surface quality. The lesions demonstrate significant dermal thickening and expansion beyond the original site of injury, consistent with hypertrophic growth patterns. There is evidence of surrounding epidermal pigmentary alterations, specifically presenting as tan to dark brown macules. These pigmentary changes appear localized to the periphery of the primary lesions, suggesting a reactive melanocytic response. No deep ulceration or active infectious processes were identified. The surrounding skin matrix shows no signs of widespread inflammation, though the focal areas of hyperpigmentation may be subject to exacerbation via UV exposure.

**IMPRESSION & MANAGEMENT PLAN**

**Primary Suspected Condition:** Keloids (Confidence: 0.85)

**Visual Clinical Presentation Analysis:**
The clinical presentation is highly characteristic of ke